In [1]:
import pandas as pd
import csv
import numpy as np
from pathlib import Path

In [55]:

verbatim = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/verbatim.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', nrows=10,
                 quoting=csv.QUOTE_NONE)

In [ ]:
list(verbatim.columns)

In [16]:
row = verbatim.iloc[-1].to_dict()

In [18]:
for k, v in row.items():
    if isinstance(v, np.ndarray):
        print(k)
        

In [30]:
multimedia = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/multimedia.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', nrows=100,
                 quoting=csv.QUOTE_NONE)

In [31]:
multimedia.columns

Index(['gbifID', 'type', 'format', 'identifier', 'references', 'title',
       'description', 'source', 'audience', 'created', 'creator',
       'contributor', 'publisher', 'license', 'rightsHolder'],
      dtype='object')

In [ ]:
multimedia.type.unique()

In [58]:
occurence = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/occurrence.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', nrows=100,
                 quoting=csv.QUOTE_NONE)
occurence.columns

Index(['gbifID', 'accessRights', 'bibliographicCitation', 'language',
       'license', 'modified', 'publisher', 'references', 'rightsHolder',
       'type',
       ...
       'publishedByGbifRegion', 'level0Gid', 'level0Name', 'level1Gid',
       'level1Name', 'level2Gid', 'level2Name', 'level3Gid', 'level3Name',
       'iucnRedListCategory'],
      dtype='object', length=223)

# Pre-existing

In [43]:
inat_audios_in_bucket = []
with open("inaturalist_audio.txt", "r") as f:
    for line in f.readlines():
        inat_audios_in_bucket.append(Path(line.strip()).stem)

In [105]:
len(inat_audios_in_bucket)

256714

In [ ]:
inat_audios_in_bucket[-50:]

In [100]:
from pathlib import Path
from urllib.parse import urlparse
import pdb

def _get_fname_from_url(url: str):
    parsed = urlparse(url)
    return Path(parsed.path).stem

In [101]:
to_download = []

In [ ]:
tot_num = 0
for i, chunk in enumerate(pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/multimedia.txt',
                 sep='\t', 
                 engine='python',  # More flexible parser
                 on_bad_lines='skip', usecols=["type","identifier"],
                 quoting=csv.QUOTE_NONE, chunksize=1_000_000)):
    
    subchunk = chunk[chunk["type"] == "Sound"]
    if not subchunk.empty:
        fnames = subchunk["identifier"].apply(_get_fname_from_url)
        fnames_bool = fnames.isin(inat_audios_in_bucket).to_numpy()

        if fnames_bool.sum() > 0:
            print(f"Found {fnames_bool.sum()} overlapping files")
            
        ids = np.where(fnames_bool == False)[0]
        fnames = subchunk["identifier"].iloc[ids].tolist()
        to_download.extend(fnames)

    if len(to_download) >= 100_000:
        tot_num += len(to_download)
        print(f"Current num of files to download = {tot_num}")
        inat_to_download = pd.DataFrame({"identifier": to_download}).to_json("inat_to_download.jsonl", orient="records",lines=True, mode="a")
        to_download = []

In [103]:
chunk.head(5)

,type,identifier
195000000,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
195000001,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
195000002,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
195000003,StillImage,https://inaturalist-open-data.s3.amazonaws.com...
195000004,StillImage,https://inaturalist-open-data.s3.amazonaws.com...


In [104]:
inat_to_download = pd.read_json("inat_to_download.jsonl", orient="records",lines=True,)
inat_to_download.shape

(406510, 1)

In [13]:
inat_to_download.head(5)

,identifier
0,https://static.inaturalist.org/sounds/17376.m4...
1,https://static.inaturalist.org/sounds/19469.mp...
2,https://static.inaturalist.org/sounds/23246.mp...
3,https://static.inaturalist.org/sounds/23384.mp...
4,https://static.inaturalist.org/sounds/29118.wa...


In [19]:
inat_to_download.head(5).to_dict()

{'identifier': {0: 'https://static.inaturalist.org/sounds/17376.m4a?1516933705',
  1: 'https://static.inaturalist.org/sounds/19469.mp3?1523741395',
  2: 'https://static.inaturalist.org/sounds/23246.mp3?1529475596',
  3: 'https://static.inaturalist.org/sounds/23384.mp3?1529678795',
  4: 'https://static.inaturalist.org/sounds/29118.wav?1547207371'}}

In [18]:
inat_to_download.identifier.apply(lambda x: Path(x).suffix).unique()

array(['.m4a?1516933705', '.mp3?1523741395', '.mp3?1529475596', ...,
       '.wav?1684085788', '.wav?1684111954', '.m4a?1660127046'],
      shape=(595470,), dtype=object)

# Double check that files downloaded are not already on bucket

In [1]:
import os
from pathlib import Path
import pandas as pd

In [15]:
inat_downloaded = list(Path("/home/gagan/esp_projects/esp-data/scripts/inat_audio_downloads/").glob("*")) # os.listdir("../inat_audio_downloads/")
len(inat_downloaded)

362500

In [16]:
extensions = set([Path(x).suffix for x in inat_downloaded])
extensions

{'.m4a', '.mp3', '.mpga', '.wav'}

In [17]:
stems_downloaded = [Path(x).stem for x in inat_downloaded]

In [18]:
new_file_stems = list(set(stems_downloaded).difference(set(inat_audios_in_bucket)))
len(new_file_stems)

326475

In [22]:
to_resample = pd.DataFrame(inat_downloaded, columns=["files"])
to_resample.head(3)

,files
0,/home/gagan/esp_projects/esp-data/scripts/inat...
1,/home/gagan/esp_projects/esp-data/scripts/inat...
2,/home/gagan/esp_projects/esp-data/scripts/inat...


In [23]:
to_resample.to_csv("../inat_to_resample_and_copy.csv", index=False)

# Metadata for new inaturalist

# We have resampled and uploaded files 
in gs://foundation-model-data/audio_16k/animalspeak2/16khz/iNaturalist (16k resampled)
Now we need to know which new ones we added

In [3]:
# old files in gs://foundation-model-data/audio_16k/animalspeak2/16khz/iNaturalist before new batch download and upload
inat_audios_in_bucket = []
with open("inaturalist_audio.txt", "r") as f:
    for line in f.readlines():
        inat_audios_in_bucket.append(Path(line.strip()).stem)

In [4]:
updated_inat_audios_in_bucket = []
with open("inat_audios_in_bucket_updated_with_new.txt", "r") as f:
    for line in f.readlines():
        updated_inat_audios_in_bucket.append(Path(line.strip()).stem)

In [5]:
len(updated_inat_audios_in_bucket)

583162

In [6]:
stems_added = list(set(updated_inat_audios_in_bucket).difference(set(inat_audios_in_bucket)))
len(stems_added)

326448

In [20]:
stems_added[-5:]

['1352550', '1019865', '1084108', '771398', '1212702']

## load darwin core archive
First inspect which columns are there ?

In [8]:
darwin = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/darwin_core/gbif-observations-dwca/observations.csv',nrows=10000)

In [9]:
darwin.columns

Index(['id', 'occurrenceID', 'basisOfRecord', 'modified', 'institutionCode',
       'collectionCode', 'datasetName', 'informationWithheld', 'catalogNumber',
       'references', 'occurrenceRemarks', 'recordedBy', 'recordedByID',
       'identifiedBy', 'identifiedByID', 'captive', 'eventDate', 'eventTime',
       'verbatimEventDate', 'verbatimLocality', 'decimalLatitude',
       'decimalLongitude', 'coordinateUncertaintyInMeters', 'geodeticDatum',
       'countryCode', 'stateProvince', 'identificationID', 'dateIdentified',
       'identificationRemarks', 'taxonID', 'scientificName', 'taxonRank',
       'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'license',
       'rightsHolder', 'inaturalistLogin', 'publishingCountry', 'projectId',
       'sex', 'lifeStage', 'reproductiveCondition', 'vitality',
       'dynamicProperties'],
      dtype='object')

In [12]:
darwin.id.sample(5)

6982    147217574
7865    147218614
2606         2579
6107     73609718
1736    110412673
Name: id, dtype: int64

In [13]:
darwin.occurrenceID.sample(5)

1811         http://www.inaturalist.org/observations/2223
4491    https://www.inaturalist.org/observations/22082...
1353    https://www.inaturalist.org/observations/22082...
8017         http://www.inaturalist.org/observations/2987
1831    https://www.inaturalist.org/observations/22082...
Name: occurrenceID, dtype: object

In [14]:
darwin.scientificName.sample(5)

2699          Bucephala clangula
521             Rostanga pulchra
7974    Desarmillaria caespitosa
4621           Banksia menziesii
5369            Rosa minutifolia
Name: scientificName, dtype: object

In [87]:
darwin_media = pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/darwin_core/gbif-observations-dwca/media.csv',nrows=10000)

## match rows of darwin core-archive to stems added to esp-ml-datasets/inaturalist/v0.1.0/raw/audio

In [2]:
import pdb

In [3]:
from pathlib import Path
from urllib.parse import urlparse
import pdb

def _get_fname_from_url(url: str):
    parsed = urlparse(url)
    return Path(parsed.path).stem

In [4]:
stems_added_raw = []
with open("files_in_esp-ml-datasets_inaturalist_v0.1.0_raw_audio.txt", "r") as f:
    for line in f.readlines():
        stems_added_raw.append(Path(line.strip()).stem)

In [5]:
stems_df = pd.Series(stems_added_raw,name="uploaded_stems")

In [6]:
stems_df.sample(10)

239689     314793
367637     743093
312282     581402
16672     1037289
355617     716626
179605    1451510
104444    1263517
367468     742725
14423     1032240
363323     733639
Name: uploaded_stems, dtype: object

In [7]:
len(stems_added_raw)

478068

In [8]:
stems_added_raw = set(stems_added_raw)

In [73]:
extensions = set()
with open("files_in_esp-ml-datasets_inaturalist_v0.1.0_raw_audio.txt", "r") as f:
    for line in f.readlines():
        extensions.add(Path(line.strip()).suffix)
extensions

{'.m4a', '.mp3', '.mpga', '.wav'}

### From the darwin core 'media' file, match for the stems that were downloaded

In [ ]:
full_data = pd.DataFrame()

for j, chunk in enumerate(pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/darwin_core/gbif-observations-dwca/media.csv', chunksize=1_000_000)):
    print(f"Processing chunk {j}")
    chunk = chunk[chunk["type"] == "Sound"]
    dups = chunk.duplicated(subset=['identifier'])
    print(f"Num duplicate identifiers = {dups.sum()}")
    chunk = chunk.drop_duplicates(subset=["identifier"])
    
    ids_added_bool = chunk["identifier"].apply(lambda x: _get_fname_from_url(x) in stems_added_raw)
    
    #ids_added_bool = ids.isin(stems_added_raw)
    if ids_added_bool.sum() == 0:
        continue 
        
    if full_data.empty:
        full_data = chunk.loc[ids_added_bool]
    else:
        full_data = pd.concat([full_data, chunk.loc[ids_added_bool]], axis=0)

    print(f"Current num found = {len(full_data)}")

if not full_data.empty:
    full_data.to_json("darwin_core_subset_new_batch_ids.jsonl", orient="records", lines=True, mode="w")

In [120]:
full_data.shape

(478688, 11)

In [122]:
full_data = full_data.drop_duplicates(subset=["identifier"])
full_data.shape

(477681, 11)

In [64]:
full_data.columns

Index(['id', 'type', 'format', 'identifier', 'references', 'created',
       'creator', 'publisher', 'license', 'rightsHolder', 'catalogNumber'],
      dtype='object')

In [10]:
full_data = pd.read_json("darwin_core_subset_new_batch_ids.jsonl",  orient="records", lines=True,)
full_data.shape

(478688, 11)

In [11]:
full_data = full_data.drop_duplicates(subset=["identifier"])
full_data.shape

(477681, 11)

In [16]:
full_data.sample(5)

,id,type,format,identifier,references,created,creator,publisher,license,rightsHolder,catalogNumber
6523,110939569,Sound,audio/mpeg,https://static.inaturalist.org/sounds/389104.m...,https://www.inaturalist.org/sounds/389104,2022-04-10T07:39:51Z,Bodo Sonnenburg,iNaturalist,http://creativecommons.org/licenses/by-nc/4.0/,Bodo Sonnenburg,389104
263117,201767214,Sound,audio/x-wav,https://static.inaturalist.org/sounds/922774.w...,https://www.inaturalist.org/sounds/922774,2024-03-09T09:23:08Z,Benjamin Schmid,iNaturalist,http://creativecommons.org/licenses/by-nc/4.0/,Benjamin Schmid,922774
269424,202218317,Sound,audio/x-wav,https://static.inaturalist.org/sounds/926609.w...,https://www.inaturalist.org/sounds/926609,2024-03-12T23:26:26Z,fredatwood,iNaturalist,http://creativecommons.org/licenses/by-nc/4.0/,fredatwood,926609
381663,212212644,Sound,audio/mpeg,https://static.inaturalist.org/sounds/1007416....,https://www.inaturalist.org/sounds/1007416,2024-04-30T11:36:11Z,Christoph Moning,iNaturalist,http://creativecommons.org/licenses/by/4.0/,Christoph Moning,1007416
349978,286618400,Sound,audio/mp4,https://static.inaturalist.org/sounds/1503606....,https://www.inaturalist.org/sounds/1503606,2025-06-03T19:24:08Z,chrisfarias,iNaturalist,http://creativecommons.org/licenses/by-nc/4.0/,chrisfarias,1503606


### now use the ids found in 'media' to fetch 'observations' data

In [ ]:
darwin_obs = pd.DataFrame()
FF = set(full_data["id"].to_list())

for j, chunk in enumerate(pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/darwin_core/gbif-observations-dwca/observations.csv', chunksize=1_000_000)):
    
    print(f"Processing chunk {j}")
    chunk = chunk.drop_duplicates()
    
    ids_added_bool = chunk["id"].apply(lambda x: x in FF) #isin(full_data["id"])
    if ids_added_bool.sum() == 0:
        continue 

    ids_added = np.where(ids_added_bool
    if darwin_obs.empty:
        darwin_obs = chunk.loc[ids_added_bool]
    else:
        darwin_obs = pd.concat([darwin_obs, chunk.loc[ids_added_bool]], axis=0)
        
    print(f"Current num found = {len(darwin_obs)}")

darwin_obs = darwin_obs.drop_duplicates()
if not darwin_obs.empty:
    darwin_obs.to_json("darwin_core_subset_new_batch_obs.jsonl", orient="records", lines=True, mode="w")

In [131]:
darwin_obs.shape

(439259, 48)

In [13]:
darwin_obs = pd.read_json("darwin_core_subset_new_batch_obs.jsonl", orient="records", lines=True,)
darwin_obs.shape

(439259, 48)

In [14]:
darwin_obs.sample(5)

,id,occurrenceID,basisOfRecord,modified,institutionCode,collectionCode,datasetName,informationWithheld,catalogNumber,references,...,license,rightsHolder,inaturalistLogin,publishingCountry,projectId,sex,lifeStage,reproductiveCondition,vitality,dynamicProperties
434254,108846201,https://www.inaturalist.org/observations/10884...,HumanObservation,2024-10-26 16:25:15+00:00,iNaturalist,Observations,iNaturalist research-grade observations,None,108846201,https://www.inaturalist.org/observations/10884...,...,http://creativecommons.org/publicdomain/zero/1.0/,Stephen John Davies,stephen220,None,https://www.inaturalist.org/projects/audio-obs...,None,None,None,alive,"{""evidenceOfPresence"":""organism""}"
388238,178430924,https://www.inaturalist.org/observations/17843...,HumanObservation,2024-07-30 20:18:50+00:00,iNaturalist,Observations,iNaturalist research-grade observations,None,178430924,https://www.inaturalist.org/observations/17843...,...,http://creativecommons.org/licenses/by-nc/4.0/,b_miller,b_miller,None,https://www.inaturalist.org/projects/audio-obs...,None,None,None,alive,"{""evidenceOfPresence"":""organism""}"
106959,192205039,https://www.inaturalist.org/observations/19220...,HumanObservation,2024-10-25 18:46:58+00:00,iNaturalist,Observations,iNaturalist research-grade observations,None,192205039,https://www.inaturalist.org/observations/19220...,...,http://creativecommons.org/licenses/by-nc/4.0/,Alexander Baransky,alexander_baransky,None,https://www.inaturalist.org/projects/audio-obs...,None,None,None,alive,"{""evidenceOfPresence"":""organism""}"
80745,80171009,https://www.inaturalist.org/observations/80171009,HumanObservation,2025-01-03 05:14:44+00:00,iNaturalist,Observations,iNaturalist research-grade observations,None,80171009,https://www.inaturalist.org/observations/80171009,...,http://creativecommons.org/licenses/by-nc/4.0/,Peter Blancher,blancherp,None,https://www.inaturalist.org/projects/reptiles-...,None,adult,None,alive,"{""evidenceOfPresence"":""organism""}"
164962,271442110,https://www.inaturalist.org/observations/27144...,HumanObservation,2025-05-07 15:19:43+00:00,iNaturalist,Observations,iNaturalist research-grade observations,None,271442110,https://www.inaturalist.org/observations/27144...,...,http://creativecommons.org/licenses/by-nc/4.0/,Lilia Efimtseva,lilia_efimtseva,None,https://www.inaturalist.org/projects/audio-obs...,None,None,None,alive,"{""evidenceOfPresence"":""organism""}"


In [17]:
from tqdm.notebook import tqdm

In [19]:
additional_cols_for_darwin = {k: [] for k in list(full_data.columns)}

for i, row in tqdm(darwin_obs.iterrows(), total=len(darwin_obs)):

    # find this row in full_data (media)
    full_data_sub = full_data[full_data["id"] == row["id"]]
    assert len(full_data_sub) > 0, "WRONG!"
    full_data_sub = full_data_sub.iloc[0].to_dict()
    
    for k in list(additional_cols_for_darwin.keys()):
        additional_cols_for_darwin[k].append(full_data_sub[k])
        

  0%|          | 0/439259 [00:00<?, ?it/s]

### Combine the obs data with the 'media' data

In [20]:
additional_cols_for_darwin = pd.DataFrame(additional_cols_for_darwin)
additional_cols_for_darwin.columns

Index(['id', 'type', 'format', 'identifier', 'references', 'created',
       'creator', 'publisher', 'license', 'rightsHolder', 'catalogNumber'],
      dtype='object')

In [21]:
set(additional_cols_for_darwin.columns).intersection(set(darwin_obs.columns))

{'catalogNumber', 'id', 'license', 'references', 'rightsHolder'}

In [23]:
additional_cols_for_darwin.drop(['catalogNumber', 'id', 'license', 'references', 'rightsHolder'], inplace=True, axis=1)
combined = pd.concat([darwin_obs, additional_cols_for_darwin], axis=1)
combined.shape

(439259, 54)

In [26]:
combined.to_json("inat_darwin_core_obs.jsonl", orient="records", lines=True)

In [29]:
assert combined["identifier"].unique().shape[0] == len(combined)

### Combine darwin core with gbif data

In [ ]:
full_df_multi = pd.DataFrame()
CC = set(combined["identifier"].to_list())

for i, multi_sub in enumerate(pd.read_csv('/home/gagan/open_audio_datasets/inaturalist_0042763-250525065834625/multimedia.txt', sep='\t', engine='python', on_bad_lines='skip', usecols=["gbifID","type","identifier"], quoting=csv.QUOTE_NONE, chunksize=1_000_000)):

    print(f"Processing chunk {i}")
    multi_sub = multi_sub[multi_sub["type"] == "Sound"].drop_duplicates(subset=["identifier"])
    
    fnames_downloaded_bool = multi_sub["identifier"].isin(CC)
    
    if fnames_downloaded_bool.sum() == 0:
        continue

    multi_sub = multi_sub.loc[fnames_downloaded_bool]

    if full_df_multi.empty:
        full_df_multi = multi_sub.copy()
    else:
        full_df_multi = pd.concat([full_df_multi, multi_sub])

    full_df_multi = full_df_multi.drop_duplicates()
    print(f"Current size of df = {len(full_df_multi)}")
    
# if not full_df_multi.empty:
#    full_df_multi.to_json("multimedia_subset.jsonl", orient="records", lines=True, mode="w")

In [34]:
full_df_multi.shape

(446381, 3)

In [35]:
full_df_multi = pd.read_json("multimedia_subset.jsonl", orient="records", lines=True)
full_df_multi.shape

(789415, 3)

In [36]:
full_df_multi = full_df_multi.drop_duplicates(subset=["identifier"])

In [37]:
full_df_multi.shape

(478052, 3)

In [38]:
full_df_multi.sample(20)

,gbifID,type,identifier
399095,5087813484,Sound,https://static.inaturalist.org/sounds/1328683....
420315,2980806117,Sound,https://static.inaturalist.org/sounds/146805.m...
128774,4937128785,Sound,https://static.inaturalist.org/sounds/488426.m...
132900,4091834123,Sound,https://static.inaturalist.org/sounds/643454.m...
53069,4875401896,Sound,https://static.inaturalist.org/sounds/1068145....
309316,4936120942,Sound,https://static.inaturalist.org/sounds/607611.m...
333975,3416229074,Sound,https://static.inaturalist.org/sounds/329769.m...
265458,3760217433,Sound,https://static.inaturalist.org/sounds/388882.w...
6470,2563472176,Sound,https://static.inaturalist.org/sounds/50664.mp...
120544,4597393138,Sound,https://static.inaturalist.org/sounds/926474.m...
